In [ ]:
from dotenv import load_dotenv
import os 
load_dotenv()
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")



## Backends and remote skill loading
Deep Agents supports different backends depending on how you want to store and manage skill files:
* StateBackend: Stores files in LangGraph agent state for the current thread.
* StoreBackend: Stores files in a LangGraph store for durable, cross-thread storage.
* FilesystemBackend: Reads and writes skill files from disk under a configurable root_dir.

## StateBackend

In [19]:
from deepagents import create_deep_agent
from deepagents.backends import StateBackend,FilesystemBackend


In [ ]:
agent = create_deep_agent(
    model="openrouter:z-ai/glm-4.7-flash",
)


In [ ]:
agent_State_backend=create_deep_agent(
    model="openrouter:openai/gpt-4o-mini",
    backend=StateBackend(
  
    )   
)


## Create the files 

In [ ]:
result=agent_State_backend.invoke(
   { 'messages': [{
        'role':'user',
        'content':(
            """Create a file at /notes/todo.txt with the following content:
            1. Buy groceries
            2. Call the bank
            3. Schedule a dentist appointment
            4. Finish the report for work"""
        )
    }]}
)


In [ ]:
result['messages'][-1].content


'The file has been created at `/notes/todo.txt` with the specified content.'

In [ ]:
result['messages']


[HumanMessage(content='Create a file at /notes/todo.txt with the following content:\n            1. Buy groceries\n            2. Call the bank\n            3. Schedule a dentist appointment\n            4. Finish the report for work', additional_kwargs={}, response_metadata={}, id='5eb62f07-ad60-4219-9d16-d06da3888018'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1791197388-5VGuSDg8uJxRcLXCuLEJ', 'created': 1791197388, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0003336, 'cost_details': {'upstream_inference_completions_cost': 2.82e-05, 'upstream_inference_prompt_cost': 0.0003054, 'upstream_inference_cost': 0.0003336}, 'system_fingerprint': 'fp_5a65effc2c'}, id='lc_run--01a10baf-2251-7e50-a012-79d36e7329d4-0', tool_calls=[{'name': 'write_file', 'args': {'file_path': '/notes/todo.txt', 'content': '1. Buy groceries\n2. Call the bank\n3. Schedule 

## Read the files

In [ ]:
followup=agent_State_backend.invoke(
    {
       "messages": result['messages']+[
           {
               "role": "user",
               "content": """read  a file at /notes/todo.txt """
           }
       ],
       "files": result.get("files", {})
    }
)


In [18]:
followup['messages'][-1].content


'Here is the content of the file at `/notes/todo.txt`:\n\n```\n1. Buy groceries\n2. Call the bank\n3. Schedule a dentist appointment\n4. Finish the report for work\n```'

## FileSystem Backend 

In [ ]:
ROOT="."
agent_files_backend=create_deep_agent(
    model="openrouter:openai/gpt-4o-mini",
    backend=FilesystemBackend(
        root_dir=ROOT,
        virtual_mode=True
    )   
)


In [ ]:
response=agent_files_backend.invoke(
   { 'messages': [{
        'role':'user',
        'content':(
            """Create a file at /notes/todo.txt with the following content:
            1. Buy groceries
            2. Call the bank
            3. Schedule a dentist appointment
            4. Finish the report for work"""
        )
    }]}
)


In [ ]:
response_read=agent_files_backend.invoke(   
    {
       "messages": result['messages']+[
           {
               "role": "user",
               "content": """read  a file at /notes/todo.txt """
           }
       ],
       "files": result.get("files", {})
    }
)


In [ ]:
response_read["messages"][-1].content


'Here is the content of the file `/notes/todo.txt`:\n\n```\n1. Buy groceries\n2. Call the bank\n3. Schedule a dentist appointment\n4. Finish the report for work\n```'

## StoreBackend verification 


In [ ]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store=InMemoryStore()

agent=create_deep_agent(
    model="openrouter:openai/gpt-4o-mini",
    backend=StoreBackend(
        namespace=lambda rt: ("user_demo"),
    ),
            store=store

)


In [ ]:
import uuid
thread_1={"configurable":{"thread_id":str(uuid.uuid4())}}
result=agent.invoke(
       { 'messages': [{
        'role':'user',
        'content':(
            """Create a file at /notes/todo.txt with the following content:
            1. Buy groceries
            2. Call the bank
            3. Schedule a dentist appointment
            4. Finish the report for work"""
        )
    }]},
    config=thread_1
)


In [ ]:
result['messages'][-1].content


'I have created the file at `/notes/todo.txt` with the specified content.'

In [ ]:
result['messages'][-1]


AIMessage(content='I have created the file at `/notes/todo.txt` with the specified content.', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1791198963-g5zVzBZaqqUUhQHyKmxO', 'created': 1791198963, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0001731, 'cost_details': {'upstream_inference_completions_cost': 1.08e-05, 'upstream_inference_prompt_cost': 0.0001623, 'upstream_inference_cost': 0.0001731}, 'system_fingerprint': 'fp_5a65effc2c'}, id='lc_run--01a10bc7-2d0f-7da2-a6e2-9ab23d8fc1c5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2106, 'output_tokens': 18, 'total_tokens': 2124, 'input_token_details': {'cache_read': 2048, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})